# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [5]:
class matrix:
    def __init__(self, *args):
        # 1 init with n and m arguments
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            self.n, self.m = args
            self.data = [[0.0 for _ in range(self.m)] for _ in range(self.n)]
        
        # 2 init with a list of lists
        elif len(args) == 1 and isinstance(args[0], list):
            input_list = args[0]
            self.n = len(input_list)
            self.m = len(input_list[0]) if self.n > 0 else 0
            
            # test matrix is properly specified
            for row in input_list:
                if len(row) != self.m:
                    raise ValueError("all rows must have the same number of columns")
            
            self.data = [row[:] for row in input_list]
        else:
            raise TypeError("invalid arguments for matrix init")

    # modified __getitem__ to support M[i][j], M[i, j], and slicing
    def __getitem__(self, index):
        if isinstance(index, tuple):
            i, j = index
            if isinstance(i, slice) or isinstance(j, slice):
                row_slice = i if isinstance(i, slice) else slice(i, i+1)
                col_slice = j if isinstance(j, slice) else slice(j, j+1)
                sub_data = [row[col_slice] for row in self.data[row_slice]]
                return matrix(sub_data)
            else:
                return self.data[i][j]
        elif isinstance(index, int) or isinstance(index, slice):
            return self.data[index]
        else:
            raise TypeError("invalid index type")

    # '=' cannot be overloaded
    def set(self, other):
        if isinstance(other, matrix):
            if self.shape() != other.shape():
                raise ValueError("matrix instances must be the same size")
            self.data = [row[:] for row in other.data]
        elif isinstance(other, list):
            n = len(other)
            m = len(other[0]) if n > 0 else 0
            if (n, m) != self.shape():
                raise ValueError("list must be the same size as the matrix")
            for row in other:
                if len(row) != m:
                    raise ValueError("all rows must have the same number of columns")
            self.data = [row[:] for row in other]
        else:
            raise TypeError("requires a matrix or list of lists")

    # 2
    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        transposed_data = [[self.data[i][j] for i in range(self.n)] for j in range(self.m)]
        return matrix(transposed_data)

    def row(self, n):
        return matrix([self.data[n]])

    def column(self, n):
        return matrix([[self.data[i][n]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        sub_data = [self.data[r][n_0:n_1] for r in range(m_0, m_1)]
        return matrix(sub_data)

    # 4 
    def scalarmul(self, c):
        new_data = [[self.data[i][j] * c for j in range(self.m)] for i in range(self.n)]
        return matrix(new_data)

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("matrices must have the same shape for addition")
        new_data = [[self.data[i][j] + N.data[i][j] for j in range(self.m)] for i in range(self.n)]
        return matrix(new_data)

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("matrices must have the same shape for subtraction")
        new_data = [[self.data[i][j] - N.data[i][j] for j in range(self.m)] for i in range(self.n)]
        return matrix(new_data)

    def mat_mult(self, N):
        if self.m != N.n:
            raise ValueError("inner dimensions must mtach for matrix multipliaction")
        new_data = [[0.0 for _ in range(N.m)] for _ in range(self.n)]
        for i in range(self.n):
            for j in range(N.m):
                for k in range(self.m):
                    new_data[i][j] += self.data[i][k] * N.data[k][j]
        return matrix(new_data)

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("matrices must have the same shape for element-wise multiplicattion")
        new_data = [[self.data[i][j] * N.data[i][j] for j in range(self.m)] for i in range(self.n)]
        return matrix(new_data)

    def equals(self, N):
        if not isinstance(N, matrix):
            return False
        if self.shape() != N.shape():
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != N.data[i][j]:
                    return False
        return True

    # 5 overload operators
    def __add__(self, N):
        return self.add(N)

    def __sub__(self, N):
        return self.sub(N)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, matrix):
            return self.mat_mult(other)
        else:
            raise TypeError("unsopported operand type for *")

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        else:
            raise TypeError("unsopported operand type for *")

    def __eq__(self, N):
        return self.equals(N)

    def __str__(self):
        return str(self.data)
    
    __repr__ = __str__

# 3
def constant(n, m, c):
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    identity = zeros(n, n)
    for i in range(n):
        identity.data[i][i] = 1.0
    return identity

# 6
if __name__ == "__main__":
    A = matrix([[1, 2], [3, 4]])
    B = matrix([[5, 6], [7, 8]])
    C = matrix([[9, 10], [11, 12]])
    I = eye(2)

    print("(AB)C == A(BC):", ((A * B) * C) == (A * (B * C)))
    print("A(B + C) == AB + AC:", (A * (B + C)) == ((A * B) + (A * C)))
    print("AB != BA:", not ((A * B) == (B * A)))
    print("AI == A:", (A * I) == A)

(AB)C == A(BC): True
A(B + C) == AB + AC: True
AB != BA: True
AI == A: True
